Take Home Exercise #2
Emma Wang

Load all required F1 datasets from the provided Databricks volume path.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

base_path = "/Volumes/gr5069/raw/f1_data"

pit_stops = spark.read.csv(f"{base_path}/pit_stops.csv", header=True, inferSchema=True)
results = spark.read.csv(f"{base_path}/results.csv", header=True, inferSchema=True)
drivers = spark.read.csv(f"{base_path}/drivers.csv", header=True, inferSchema=True)
races = spark.read.csv(f"{base_path}/races.csv", header=True, inferSchema=True)

spark.read.csv() loads structured data  inferSchema=True automatically detects column types

Q1 — Average Pit Stop Time

Group by race and driver, then compute:

average pit stop time
fastest pit stop
slowest pit stop

In [0]:
pit_stats = pit_stops.groupBy("raceId", "driverId").agg(
    avg("milliseconds").alias("avg_pit_time"),
    min("milliseconds").alias("fastest_pit"),
    max("milliseconds").alias("slowest_pit")
)

display(pit_stats)

groupBy() groups data per race and driver
avg() calculates average pit stop time
min() and max() identify extreme values

Alternative

Use Spark SQL instead of DataFrame API.

Q2 — Rank Pit Stop Time

Join pit stop data with race results
Remove drivers who did not finish (DNF)
Rank drivers by pit stop time within each race

In [0]:
joined = pit_stats.join(results, ["raceId", "driverId"])

filtered = joined.filter(col("positionOrder").isNotNull())

window_spec = Window.partitionBy("raceId").orderBy("avg_pit_time")

ranked = filtered.withColumn("pit_rank", rank().over(window_spec))

display(ranked)

join() merges datasets
filter() removes invalid race results
Window.partitionBy() creates race-level ranking
rank() assigns ranking

Alternative

Use dense_rank() for continuous ranking.

Q3 — Insert Missing Driver Codes

Identify missing driver codes
Fill using first 3 letters of surname
Convert to uppercase

In [0]:
drivers_updated = drivers.withColumn(
    "code",
    when(col("code").isNull(),
         upper(substring(col("surname"), 1, 3)))
    .otherwise(col("code"))
)

display(drivers_updated)

Explanation
when() applies conditional logic
substring() extracts characters
upper() standardizes format

Alternative

Use initials from full name.

Q4 — Youngest & Oldest Driver

Define age as number of full years at race date
Join drivers and races
Identify youngest and oldest per race

In [0]:
df = results.join(drivers, "driverId").join(races, "raceId")

df = df.withColumn(
    "Age",
    floor(datediff(col("date"), col("dob")) / 365)
)

young_window = Window.partitionBy("raceId").orderBy("Age")
old_window = Window.partitionBy("raceId").orderBy(col("Age").desc())

youngest = df.withColumn("rank", rank().over(young_window)).filter("rank = 1")
oldest = df.withColumn("rank", rank().over(old_window)).filter("rank = 1")

display(youngest)
display(oldest)

Explanation
datediff() calculates days difference
dividing by 365 converts to years
Window identifies min/max age

Q5 — Wins & Non-Wins Before Each Race

For each driver:

count wins BEFORE current race
count non-win finishes BEFORE current race

In [0]:
df = results.withColumn(
    "win_flag", when(col("positionOrder") == 1, 1).otherwise(0)
).withColumn(
    "non_win_flag",
    when((col("positionOrder") > 1) & col("positionOrder").isNotNull(), 1).otherwise(0)
)

window_spec = Window.partitionBy("driverId").orderBy("raceId") \
    .rowsBetween(Window.unboundedPreceding, -1)

df = df.withColumn(
    "wins_before",
    sum("win_flag").over(window_spec)
).withColumn(
    "non_wins_before",
    sum("non_win_flag").over(window_spec)
)

display(df)

Explanation
rowsBetween(..., -1) ensures current race is excluded
cumulative sum tracks history

Alternative

Use lag() for previous race analysis.

Q6 — Own Question

Question

Which constructor is most consistent across races?

Logic

Use variance of finishing positions to measure consistency.

In [0]:
consistency = results.groupBy("constructorId").agg(
    avg("positionOrder").alias("avg_position"),
    stddev("positionOrder").alias("position_variance")
)

display(consistency)

Explanation

Lower variance indicates more consistent performance.